In [13]:
print(ROOT)
print(sorted(p.relative_to(ROOT).as_posix() for p in (ROOT / "data/gold").glob("**/*") if p.is_file()))

c:\Users\imbir\OneDrive\Documentos\Nova pasta (2)\enem-aneel-pipeline
['data/gold/ml_ready/enem_aneel_ml_ready.parquet']


In [14]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.append(str(ROOT))
print("raiz:", ROOT)

raiz: c:\Users\imbir\OneDrive\Documentos\Nova pasta (2)\enem-aneel-pipeline


In [15]:
j = pd.read_parquet(ROOT / "data/silver/joined/silver_joined.parquet")
esperadas = ["codigo_municipio", "NomMunicipio", "ano", "mes", "dec_horas_mensal",
             "dec_horas_max", "fec_freq_mensal", "total_inscritos", "total_presentes",
             "total_abstencao", "taxa_abstencao", "taxa_presenca"]
print("faltando:", [c for c in esperadas if c not in j.columns])

faltando: ['dec_horas_max', 'total_presentes', 'taxa_presenca']


In [16]:
enem = pd.read_parquet(ROOT / "data/silver/enem/enem_silver.parquet")
print(enem.groupby("ano").agg(
    inscritos=("total_inscritos", "sum"),
    abstencao_media=("taxa_abstencao", "mean"),
    municipios_abaixo_30=("total_inscritos", lambda s: (s < 30).sum()),
))

      inscritos  abstencao_media  municipios_abaixo_30
ano                                                   
2020      20235         0.334555                     0
2021      20806         0.342866                     0
2022      20363         0.342975                     0
2023      20178         0.336140                     0
2024      20338         0.336260                     0


In [17]:
for lim in [0.25, 0.30, 0.35, 0.40]:
    print(lim, (enem["taxa_abstencao"] > lim).groupby(enem["ano"]).mean().round(2).to_dict())

0.25 {2020: 0.83, 2021: 0.88, 2022: 0.83, 2023: 0.83, 2024: 0.81}
0.3 {2020: 0.64, 2021: 0.64, 2022: 0.69, 2023: 0.64, 2024: 0.62}
0.35 {2020: 0.44, 2021: 0.48, 2022: 0.51, 2023: 0.43, 2024: 0.44}
0.4 {2020: 0.24, 2021: 0.3, 2022: 0.24, 2023: 0.23, 2024: 0.24}


In [18]:
an = pd.read_parquet(ROOT / "data/silver/aneel/aneel_silver.parquet")
jan_jul = an[an["mes"] <= 7].groupby(["codigo_municipio", "ano"])["mes"].nunique()
print("município-ano com janela jan-jul incompleta:", (jan_jul < 7).sum())

município-ano com janela jan-jul incompleta: 34


In [19]:
import json
print(json.dumps(json.load(open(ROOT / "data/gold/analytics/gold_metricas.json")), indent=2, ensure_ascii=False))
exc = pd.read_parquet(ROOT / "data/gold/analytics/coorte_excluidos.parquet")
print(exc.groupby(["_motivo_exclusao", "ano"]).size())

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\imbir\\OneDrive\\Documentos\\Nova pasta (2)\\enem-aneel-pipeline\\data\\gold\\analytics\\gold_metricas.json'